In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Combined One-vs-Rest (OvR) 5-Logistic Regressor Pipeline in R

This notebook trains **5 dedicated binary Logistic Regression models (`glm(family = binomial)`)** for each ESI triage level, then **combines them into one unified multi-class OvR predictor object** and benchmarks its performance.

### Key Highlights:
1. **Domain Feature Engineering**: Calculates **Shock Index**, **Vital Ranges**, and **Age-Adjusted Instability** indices.
2. **5 Binary Regressors**: Fits 5 One-vs-Rest models ($P(\text{ESI} = 1)$, $P(\text{ESI} = 2)$, $P(\text{ESI} = 3)$, $P(\text{ESI} = 4)$, $P(\text{ESI} = 5)$).
3. **Unified OvR Ensemble Object**: Wraps all 5 binary models into a single combined model instance with probability and class prediction interfaces.
4. **Comprehensive Benchmarking**: Evaluates the combined ensemble on Validation and Test sets (**ROC-AUC**, **Accuracy**, **Macro Precision**, **Confusion Matrix**).

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
library(jsonlite)
library(caret)
library(dplyr)
library(ggplot2)
library(pROC)
library(e1071)

# Paths to config files
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}

hyper_path <- "../config/hyper_optimize.json"
if (!file.exists(hyper_path)) {
  hyper_path <- "config/hyper_optimize.json"
}

# Parse JSON configs
config <- fromJSON(config_path)
hyper_config <- if (file.exists(hyper_path)) fromJSON(hyper_path) else list()

cat("=== Configuration Loaded ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Target Classes:  ", paste(config$classes$outputs, collapse = ", "), "\n")
cat("Features Count:  ", length(config$features$data_name), "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Full Unbalanced Data & Feature Engineering
# ---------------------------------------------------------
set.seed(config$training$random_state)

# Determine relative path for datasets
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}

cat("Loading dataset from:", data_file, "...\n")

# Load into an isolated environment
data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))

raw_df <- get(data_obj_name, envir = data_env)

target_col <- config$classes$target_col
feature_cols <- config$features$data_name

# ---------------------------------------------------------
# Feature Engineering: Shock Index, Vital Ranges, Age Instability
# ---------------------------------------------------------
if ("pulse_last" %in% names(raw_df) && "sbp_last" %in% names(raw_df)) {
  raw_df$shock_index <- raw_df$pulse_last / (raw_df$sbp_last + 1e-5)
}
if ("pulse_median" %in% names(raw_df) && "sbp_median" %in% names(raw_df)) {
  raw_df$shock_index_median <- raw_df$pulse_median / (raw_df$sbp_median + 1e-5)
}
if ("pulse_max" %in% names(raw_df) && "pulse_min" %in% names(raw_df)) {
  raw_df$pulse_range <- raw_df$pulse_max - raw_df$pulse_min
}
if ("sbp_max" %in% names(raw_df) && "sbp_min" %in% names(raw_df)) {
  raw_df$sbp_range <- raw_df$sbp_max - raw_df$sbp_min
}
if ("resp_max" %in% names(raw_df) && "resp_min" %in% names(raw_df)) {
  raw_df$resp_range <- raw_df$resp_max - raw_df$resp_min
}
if ("spo2_max" %in% names(raw_df) && "spo2_min" %in% names(raw_df)) {
  raw_df$spo2_range <- raw_df$spo2_max - raw_df$spo2_min
}
if ("age" %in% names(raw_df)) {
  raw_df$age_risk_factor <- ifelse(raw_df$age > 65, 1.5, ifelse(raw_df$age < 18, 1.3, 1.0))
  if ("shock_index" %in% names(raw_df)) {
    raw_df$age_adjusted_shock_index <- raw_df$shock_index * raw_df$age_risk_factor
  }
  pulse_val <- if ("pulse_last" %in% names(raw_df)) raw_df$pulse_last else 75
  sbp_val   <- if ("sbp_last" %in% names(raw_df)) raw_df$sbp_last else 120
  spo2_val  <- if ("spo2_min" %in% names(raw_df)) raw_df$spo2_min else 98
  pulse_rng <- if ("pulse_range" %in% names(raw_df)) raw_df$pulse_range else 0
  sbp_rng   <- if ("sbp_range" %in% names(raw_df)) raw_df$sbp_range else 0
  
  raw_df$age_adjusted_instability <- (
    (ifelse(pulse_val > 100 | pulse_val < 50, 1.5, 1.0) * raw_df$age_risk_factor) +
    (ifelse(sbp_val < 90 | sbp_val > 160, 1.5, 1.0) * raw_df$age_risk_factor) +
    (ifelse(spo2_val < 92, 2.0, 1.0)) +
    (pulse_rng * 0.05 + sbp_rng * 0.05)
  )
}

engineered_cols <- c("shock_index", "shock_index_median", "pulse_range", "sbp_range", "resp_range", "spo2_range", "age_risk_factor", "age_adjusted_shock_index", "age_adjusted_instability")
all_feature_cols <- unique(c(feature_cols, intersect(engineered_cols, names(raw_df))))

selected_cols <- intersect(c(all_feature_cols, target_col), names(raw_df))
df <- raw_df[, selected_cols, drop = FALSE]

if (!is.null(config$features$data_string_list)) {
  for (cat_var in names(config$features$data_string_list)) {
    if (cat_var %in% names(df)) {
      df[[cat_var]] <- factor(df[[cat_var]], levels = config$features$data_string_list[[cat_var]])
    }
  }
}

target_classes <- as.character(config$classes$outputs)
df[[target_col]] <- factor(df[[target_col]], levels = target_classes)

if (any(is.na(df))) {
  df <- na.omit(df)
}

cat(sprintf("Processed dataset ready: %d rows x %d cols\n", nrow(df), ncol(df)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Data Partitioning & Preprocessing
# ---------------------------------------------------------
set.seed(config$training$random_state)

test_size <- config$training$test_size
val_size  <- config$training$val_size

in_train_val <- createDataPartition(df[[target_col]], p = 1 - test_size, list = FALSE)
train_val_df <- df[in_train_val, ]
test_df      <- df[-in_train_val, ]

rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df[[target_col]], p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]

# Preprocess numeric features (center and scale)
numeric_cols <- names(train_df)[sapply(train_df, is.numeric)]
preproc <- preProcess(train_df[, numeric_cols], method = c("center", "scale"))

train_df[, numeric_cols] <- predict(preproc, train_df[, numeric_cols])
val_df[, numeric_cols]   <- predict(preproc, val_df[, numeric_cols])
test_df[, numeric_cols]  <- predict(preproc, test_df[, numeric_cols])

cat(sprintf("Partition sizes:\n  Train: %d rows (%.1f%%)\n  Val:   %d rows (%.1f%%)\n  Test:  %d rows (%.1f%%)\n",
            nrow(train_df), nrow(train_df)/nrow(df)*100,
            nrow(val_df), nrow(val_df)/nrow(df)*100,
            nrow(test_df), nrow(test_df)/nrow(df)*100))

cat("Train Class Distribution:\n")
print(table(train_df[[target_col]]))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train 5 Dedicated One-vs-Rest Binary Logistic Regression Models
# ---------------------------------------------------------
set.seed(config$training$random_state)

models_list <- list()
cat("Training 5 Dedicated Binary Logistic Regression Models...\n")

for (cls in target_classes) {
  cat(sprintf(" -> Training Model for ESI Level %s (1 = ESI %s, 0 = Rest)...\n", cls, cls))
  
  binary_train <- train_df
  binary_train$binary_target <- ifelse(as.character(binary_train[[target_col]]) == cls, 1, 0)
  
  feat_names <- setdiff(names(binary_train), c(target_col, "binary_target"))
  binary_formula <- as.formula(paste("binary_target ~", paste(feat_names, collapse = " + ")))
  
  model_cls <- glm(formula = binary_formula, data = binary_train, family = binomial(link = "logit"))
  models_list[[cls]] <- model_cls
}

cat("\nAll 5 Binary Logistic Regression models trained successfully!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Combine 5 Binary OvR Models into One Unified Predictor Object
# ---------------------------------------------------------

# Constructor function for the combined OvR Logistic Model object
create_ovr_logistic_model <- function(models_list, target_classes) {
  obj <- list(
    models = models_list,
    target_classes = target_classes
  )
  class(obj) <- "ovr_logistic_model"
  return(obj)
}

# S3 predict method for ovr_logistic_model class
predict.ovr_logistic_model <- function(object, newdata, type = "class") {
  target_classes <- object$target_classes
  prob_matrix <- matrix(0, nrow = nrow(newdata), ncol = length(target_classes))
  colnames(prob_matrix) <- target_classes
  
  for (cls in target_classes) {
    model <- object$models[[cls]]
    prob_matrix[, cls] <- predict(model, newdata = newdata, type = "response")
  }
  
  if (type %in% c("prob", "probabilities", "raw")) {
    return(prob_matrix)
  } else {
    max_idx <- max.col(prob_matrix, ties.method = "first")
    pred_classes <- factor(target_classes[max_idx], levels = target_classes)
    return(pred_classes)
  }
}

# Instantiate the single unified combined model
combined_ovr_model <- create_ovr_logistic_model(models_list, target_classes)
cat("=== Unified Combined OvR Logistic Regressor Model Created Successfully ===\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Benchmark Combined OvR Model Object on Validation & Test Sets
# ---------------------------------------------------------

benchmark_combined_model <- function(combined_model, data, set_name, target_col, target_classes) {
  # 1. Predict class probabilities and class predictions using unified predict method
  prob_matrix <- predict(combined_model, newdata = data, type = "prob")
  pred_factor <- predict(combined_model, newdata = data, type = "class")
  actual_factor <- factor(data[[target_col]], levels = target_classes)
  
  # 2. Confusion Matrix & Overall Accuracy
  cm <- confusionMatrix(pred_factor, actual_factor)
  acc <- cm$overall["Accuracy"]
  
  # 3. Macro & Per-Class Precision
  precision_vec <- if (is.matrix(cm$byClass)) cm$byClass[, "Pos Pred Value"] else cm$byClass["Pos Pred Value"]
  macro_precision <- mean(precision_vec, na.rm = TRUE)
  
  # 4. Multi-Class ROC-AUC (Hand & Till, 2001)
  roc_auc <- tryCatch({
    as.numeric(pROC::multiclass.roc(actual_factor, prob_matrix)$auc)
  }, error = function(e) NA)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("  COMBINED OvR LOGISTIC MODEL - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Multi-Class ROC-AUC : %.4f\n", roc_auc))
  cat(sprintf("  Overall Accuracy    : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision     : %.4f\n", macro_precision))
  cat("\n  Precision by ESI Class:\n")
  for (cls in names(precision_vec)) {
    cat(sprintf("    %-15s : %.4f\n", cls, precision_vec[cls]))
  }
  cat("\nFull Confusion Matrix:\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
}

# Benchmark Combined Model on Validation Set
benchmark_combined_model(combined_ovr_model, val_df, "Validation", target_col, target_classes)

# Benchmark Combined Model on Test Set
benchmark_combined_model(combined_ovr_model, test_df, "Test", target_col, target_classes)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save Combined OvR Model Object Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)

# Save single combined model object
combined_model_path <- file.path(deploy_dir, "combined_ovr_logistic_model.rds")
saveRDS(combined_ovr_model, file = combined_model_path)
cat(sprintf("Saved Unified Combined OvR Logistic Regressor model to: %s\n", combined_model_path))